# 工具审批

**常见用法**：敏感操作（支付、发布、删除、发邮件）执行前人工审批（HITL）；
审批逻辑内聚在钩子里，换图挂上即生效，图结构零改动。

**钩子内的做法**：
- 名单外的工具直接 `execute(request)` 放行；名单内的调 `interrupt(payload)` 挂起等决策
- 批准 → `execute(request)` 放行；拒绝 → **不调 execute**，回填 `status="error"` 的 ToolMessage（每个 tool_call_id 必须有一条应答）
- 坑：钩子路径下 `handle_tool_errors=True` 会吞 `GraphInterrupt`，需自定义 handler 放行（见第 3 节与 CheatSheet 7.4）

In [10]:
from typing import Literal

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode
from langgraph.prebuilt.tool_node import ToolCallRequest  # 注意：不在 prebuilt 顶层导出
from langgraph.types import Command, interrupt
from rich import print

load_dotenv(override=True)

model = init_chat_model(
    model_provider="deepseek",
    model="deepseek-flash",
    extra_body={"thinking": {"type": "disabled"}}
)


# 定义工具
@tool(parse_docstring=True)
def get_weather(city: str) -> str:
    """根据城市名称查询天气

    Args:
        city (str): 城市名称

    Returns:
        str: 城市天气情况
    """
    return f"{city} 的天气是晴天，温度 25°C"


@tool(parse_docstring=True)
def get_news(topic: Literal["科技", "体育", "娱乐"]) -> str:
    """根据主题查询新闻

    Args:
        topic (Literal["科技", "体育", "娱乐"]): 新闻主题

    Returns:
        str: 新闻内容
    """
    return {
        "科技": "最新科技新闻：AI 技术正在快速发展。",
        "体育": "最新体育新闻：中国队取得了胜利。",
        "娱乐": "最新娱乐新闻：歌手发布了新专辑。",
    }[topic]


tools = [get_weather, get_news]
model_with_tools = model.bind_tools(tools)


class ChatState(MessagesState):
    pass


def wrap_tool_call(request: ToolCallRequest, execute) -> ToolMessage | object:
    """工具审批钩子：每个工具调用执行前都会经过这里"""
    tc = request.tool_call

    # ===== 挂起，等用户决策：True 放行 / False 拒绝 =====
    approved = interrupt({
        "name": tc["name"],
        "args": tc["args"],
        "tool_call_id": tc["id"],
    })

    if approved:
        return execute(request)  # 批准：执行原工具
    # 拒绝：不执行工具，回填错误信息给模型
    return ToolMessage(
        name=tc["name"],
        content=f"用户拒绝了 `{tc['name']}` 的调用，工具未执行。请不要重试，直接向用户说明。",
        tool_call_id=tc["id"],
        status="error"
    )


# 工具节点：钩子接管审批，错误按 handler 策略处理
tool_node = ToolNode(tools, wrap_tool_call=wrap_tool_call)


# 定义状态
class State(MessagesState):
    pass


# 模型节点
def llm_node(state: ChatState) -> State:
    ai_msg = model_with_tools.invoke(state["messages"])
    return {"messages": [ai_msg]}


def router(state: ChatState) -> Literal["tool_node", END]:
    return "tool_node" if state["messages"][-1].tool_calls else END


builder = StateGraph(state_schema=ChatState)
builder.add_node("llm_node", llm_node)
builder.add_node("tool_node", tool_node)
builder.add_edge(START, "llm_node")
builder.add_conditional_edges("llm_node", router, [END, "tool_node"])
builder.add_edge("tool_node", "llm_node")

checkpointer = InMemorySaver()
config = {"configurable": {"thread_id": "1"}}
graph = builder.compile(checkpointer)
res = graph.invoke({"messages": [HumanMessage("帮我查一下北京的天气,以及科技和体育方面的新闻")]}, config=config)
print(res)

{
    'messages': [
        HumanMessage(
            content='帮我查一下北京的天气,以及科技和体育方面的新闻',
            additional_kwargs={},
            response_metadata={},
            id='33e2f4f1-b653-4bf1-82b0-1985dfc41d7c'
        ),
        AIMessage(
            content='我来帮您查询北京天气以及科技、体育新闻。',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 107,
                    'prompt_tokens': 348,
                    'total_tokens': 455,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 128,
                        'image_tokens': None,
                        'text_tokens': None
                    },
                    'prompt_cache_hit_tokens': 128,
                    'prompt_cache_miss_tokens': 220
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-flash',
                'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669',
                'id': '23ab3744-fd27-470f-a9aa-7a0ff5e9a057',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a0b0b6-d609-7a21-839c-7b896c2ce1f3-0',
            tool_calls=[
                {
                    'name': 'get_weather',
                    'args': {'city': '北京'},
                    'id': 'call_00_1zoszvB3bKUgSos7J3Fl5535',
                    'type': 'tool_call'
                },
                {
                    'name': 'get_news',
                    'args': {'topic': '科技'},
                    'id': 'call_01_ByKoslOys5KyATcEdQMz3527',
                    'type': 'tool_call'
                },
                {
                    'name': 'get_news',
                    'args': {'topic': '体育'},
                    'id': 'call_02_qXI2heGtOB9has7bqpeQ2470',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 348,
                'output_tokens': 107,
                'total_tokens': 455,
                'input_token_details': {'cache_read': 128},
                'output_token_details': {}
            }
        )
    ],
    '__interrupt__': [
        Interrupt(
            value={
                'name': 'get_weather',
                'args': {'city': '北京'},
                'tool_call_id': 'call_00_1zoszvB3bKUgSos7J3Fl5535'
            },
            id='07e8fc777ea75a2ac2270ab4438373cd'
        )
    ]
}

In [11]:
while res.get("__interrupt__"):
    info = res["__interrupt__"][0].value
    ans = input(f"允许 {info['name']}({info['args']})? (y/n) ").strip().lower() in ("y", "yes", "是", "1")
    res = graph.invoke(Command(resume=ans), config=config)
print(res)

{
    'messages': [
        HumanMessage(
            content='帮我查一下北京的天气,以及科技和体育方面的新闻',
            additional_kwargs={},
            response_metadata={},
            id='33e2f4f1-b653-4bf1-82b0-1985dfc41d7c'
        ),
        AIMessage(
            content='我来帮您查询北京天气以及科技、体育新闻。',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 107,
                    'prompt_tokens': 348,
                    'total_tokens': 455,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 128,
                        'image_tokens': None,
                        'text_tokens': None
                    },
                    'prompt_cache_hit_tokens': 128,
                    'prompt_cache_miss_tokens': 220
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-flash',
                'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669',
                'id': '23ab3744-fd27-470f-a9aa-7a0ff5e9a057',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a0b0b6-d609-7a21-839c-7b896c2ce1f3-0',
            tool_calls=[
                {
                    'name': 'get_weather',
                    'args': {'city': '北京'},
                    'id': 'call_00_1zoszvB3bKUgSos7J3Fl5535',
                    'type': 'tool_call'
                },
                {
                    'name': 'get_news',
                    'args': {'topic': '科技'},
                    'id': 'call_01_ByKoslOys5KyATcEdQMz3527',
                    'type': 'tool_call'
                },
                {
                    'name': 'get_news',
                    'args': {'topic': '体育'},
                    'id': 'call_02_qXI2heGtOB9has7bqpeQ2470',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 348,
                'output_tokens': 107,
                'total_tokens': 455,
                'input_token_details': {'cache_read': 128},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content='北京 的天气是晴天，温度 25°C',
            name='get_weather',
            id='ad5cd0fc-493f-45e2-a2b4-689aa8052f78',
            tool_call_id='call_00_1zoszvB3bKUgSos7J3Fl5535'
        ),
        ToolMessage(
            content='最新科技新闻：AI 技术正在快速发展。',
            name='get_news',
            id='ed75d8de-6798-41de-ad41-8b68db50dcc7',
            tool_call_id='call_01_ByKoslOys5KyATcEdQMz3527'
        ),
        ToolMessage(
            content='最新体育新闻：中国队取得了胜利。',
            name='get_news',
            id='6adc7549-fda0-41a3-a3cc-b2540c19d494',
            tool_call_id='call_02_qXI2heGtOB9has7bqpeQ2470'
        ),
        AIMessage(
            content='好的，为您查询到以下信息：\n\n**🌤️ 北京天气**\n- 天气：晴天\n- 温度：25°C\n- 
天气不错，适合外出活动～\n\n**📱 科技新闻**\n- AI 技术正在快速发展。\n\n**🏆 体育新闻**\n- 
中国队取得了胜利。\n\n需要我再帮您查询其他城市天气或其他类型的新闻吗？',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 79,
                    'prompt_tokens': 511,
                    'total_tokens': 590,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 256,
                        'image_tokens': None,
                        'text_tokens': None
                    },
                    'prompt_cache_hit_tokens': 256,
                    'prompt_cache_miss_tokens': 255
                },
          